# AI Programming — Lecture 19
## Lab 4-5: Encoder–Decoder Transformer for ETTh1 Forecasting
### Auxiliary Context + Residual Prediction

이번에는 Transformer의 **Encoder–Decoder 구조**를 시계열 예측에 적용합니다.

- Encoder input: `HUFL, HULL, MUFL, MULL, LUFL, LULL`
- Decoder input: `OT`
- Target: 미래 `OT`
- Prediction: residual autoregressive forecasting

### 핵심 아이디어
```text
Auxiliary variables
→ Transformer Encoder
→ context representations (K, V)

Past / generated OT
→ Transformer Decoder
→ Causal Self-Attention
→ Cross-Attention to Encoder
→ Residual Prediction
```

### 학습 목표
- Encoder와 Decoder가 서로 다른 입력을 처리하는 구조를 이해합니다.
- Decoder의 causal self-attention과 cross-attention을 구분합니다.
- Auxiliary context가 OT forecasting에 어떻게 사용되는지 확인합니다.

In [ ]:
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("keras.ops:", hasattr(keras, "ops"))
print("GPU:", gpus)
from google.colab import drive

drive.mount('/content/drive')


## 1. ETTh1 데이터 불러오기

In [ ]:
# Colab example:
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv'

df = pd.read_csv(DATA_PATH)

target_column = 'OT'
aux_columns = [
    c for c in df.columns
    if c not in ['date', target_column]
]

print("Shape:", df.shape)
print("Auxiliary columns:", aux_columns)
print(df.head())

ot = df[[target_column]].values.astype('float32')
aux = df[aux_columns].values.astype('float32')


## 2. Chronological Split과 Standardization

In [ ]:
CONTEXT_LEN = 96
PRED_LEN = 24
DECODER_LEN = CONTEXT_LEN + PRED_LEN - 1   # 119

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

ot_train_raw = ot[:train_end]
ot_val_raw = ot[train_end:val_end]
ot_test_raw = ot[val_end:]

aux_train_raw = aux[:train_end]
aux_val_raw = aux[train_end:val_end]
aux_test_raw = aux[val_end:]

ot_scaler = StandardScaler()
aux_scaler = StandardScaler()

ot_train = ot_scaler.fit_transform(ot_train_raw)
ot_val = ot_scaler.transform(ot_val_raw)
ot_test = ot_scaler.transform(ot_test_raw)

aux_train = aux_scaler.fit_transform(aux_train_raw)
aux_val = aux_scaler.transform(aux_val_raw)
aux_test = aux_scaler.transform(aux_test_raw)

print("Train:", ot_train.shape, aux_train.shape)
print("Validation:", ot_val.shape, aux_val.shape)
print("Test:", ot_test.shape, aux_test.shape)

## 3. Encoder–Decoder Sequence 구성

Encoder에는 과거 96 step의 6개 auxiliary variable이 들어갑니다.

Decoder에는 과거 OT와 teacher-forcing용 target sequence가 들어갑니다.

In [ ]:
def create_sequences(aux_values, ot_values, context_len=96, pred_len=24):
    total_len = context_len + pred_len

    X_enc, X_dec, Y, Y_res = [], [], [], []

    for i in range(len(ot_values) - total_len + 1):
        ot_window = ot_values[i:i + total_len]

        # Encoder: auxiliary variables from past 96 steps only
        X_enc.append(
            aux_values[i:i + context_len]
        )

        # Decoder: past 96 OT + first 23 future OT values
        X_dec.append(
            ot_window[:-1]
        )

        # Absolute future OT
        future = ot_window[context_len:]
        Y.append(future)

        # Fixed residual baseline = last observed OT
        baseline = ot_window[context_len - 1]
        Y_res.append(future - baseline)

    return (
        np.array(X_enc, dtype='float32'),
        np.array(X_dec, dtype='float32'),
        np.array(Y, dtype='float32'),
        np.array(Y_res, dtype='float32')
    )

X_enc_train, X_dec_train, y_train, y_train_res = create_sequences(
    aux_train, ot_train, CONTEXT_LEN, PRED_LEN
)
X_enc_val, X_dec_val, y_val, y_val_res = create_sequences(
    aux_val, ot_val, CONTEXT_LEN, PRED_LEN
)
X_enc_test, X_dec_test, y_test, y_test_res = create_sequences(
    aux_test, ot_test, CONTEXT_LEN, PRED_LEN
)

print("Encoder train:", X_enc_train.shape)
print("Decoder train:", X_dec_train.shape)
print("Target train :", y_train_res.shape)
print("Encoder val  :", X_enc_val.shape)
print("Encoder test :", X_enc_test.shape)

## 4. Learned Positional Embedding

In [ ]:
class LearnedPositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()
        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, inputs):
        positions = keras.ops.arange(
            0, keras.ops.shape(inputs)[1], 1
        )
        return inputs + self.position_embedding(positions)

## 5. Transformer Encoder Block

In [ ]:
class EncoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(ff_dim, activation='relu')
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=None):
        attention_output = self.attention(
            inputs,
            inputs,
            training=training
        )

        x = self.norm1(
            inputs + self.dropout1(attention_output, training=training)
        )

        ffn_output = self.dense2(self.dense1(x))

        return self.norm2(
            x + self.dropout2(ffn_output, training=training)
        )

## 6. Transformer Decoder Block

Decoder에는 두 종류의 attention이 있습니다.

1. **Causal Self-Attention**: 이전 OT sequence를 참조
2. **Cross-Attention**: Encoder가 만든 auxiliary context를 참조

In [ ]:
class DecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.self_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.cross_attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(ff_dim, activation='relu')
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.norm3 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
        self.dropout3 = layers.Dropout(dropout)

    def call(self, inputs, encoder_outputs, training=None):
        self_attention_output = self.self_attention(
            inputs,
            inputs,
            use_causal_mask=True,
            training=training
        )

        x = self.norm1(
            inputs
            + self.dropout1(self_attention_output, training=training)
        )

        cross_attention_output = self.cross_attention(
            query=x,
            value=encoder_outputs,
            key=encoder_outputs,
            training=training
        )

        x = self.norm2(
            x
            + self.dropout2(cross_attention_output, training=training)
        )

        ffn_output = self.dense2(self.dense1(x))

        return self.norm3(
            x + self.dropout3(ffn_output, training=training)
        )

## 7. Encoder–Decoder Transformer 구성

In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

# ----- Encoder -----
encoder_inputs = keras.Input(
    shape=(CONTEXT_LEN, len(aux_columns)),
    name='encoder_inputs'
)

encoder_projection = layers.Dense(EMBED_DIM)
enc = encoder_projection(encoder_inputs)

encoder_position = LearnedPositionalEmbedding(
    CONTEXT_LEN, EMBED_DIM
)
enc = encoder_position(enc)

encoder1 = EncoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
enc = encoder1(enc)

encoder2 = EncoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
encoder_outputs = encoder2(enc)

# ----- Decoder -----
decoder_inputs = keras.Input(
    shape=(DECODER_LEN, 1),
    name='decoder_inputs'
)

decoder_projection = layers.Dense(EMBED_DIM)
dec = decoder_projection(decoder_inputs)

decoder_position = LearnedPositionalEmbedding(
    DECODER_LEN, EMBED_DIM
)
dec = decoder_position(dec)

decoder1 = DecoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
dec = decoder1(dec, encoder_outputs)

decoder2 = DecoderBlock(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
dec = decoder2(dec, encoder_outputs)

residual_head = layers.Dense(1)
all_residuals = residual_head(dec)

forecast_layer = layers.Lambda(
    lambda x: x[:, -PRED_LEN:, :]
)
forecast_residuals = forecast_layer(all_residuals)

model = keras.Model(
    [encoder_inputs, decoder_inputs],
    forecast_residuals,
    name='encoder_decoder_residual_forecaster'
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='mse',
    metrics=['mae']
)

model.summary()

## 8. Teacher Forcing으로 학습

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    [X_enc_train, X_dec_train],
    y_train_res,
    validation_data=(
        [X_enc_val, X_dec_val],
        y_val_res
    ),
    epochs=100,
    batch_size=64,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.xlabel('Epoch')
plt.ylabel('Residual MSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()

## 9. 빠른 Inference용 Encoder / Decoder Model 구성

In [ ]:
# Encoder model
encoder_model = keras.Model(
    encoder_inputs,
    encoder_outputs,
    name='encoder_model'
)

# Reuse trained decoder layers with a precomputed encoder context
decoder_sequence_input = keras.Input(
    shape=(DECODER_LEN, 1),
    name='decoder_sequence_input'
)

encoded_context_input = keras.Input(
    shape=(CONTEXT_LEN, EMBED_DIM),
    name='encoded_context_input'
)

dec_inf = decoder_projection(decoder_sequence_input)
dec_inf = decoder_position(dec_inf)
dec_inf = decoder1(dec_inf, encoded_context_input)
dec_inf = decoder2(dec_inf, encoded_context_input)
residuals_inf = residual_head(dec_inf)

decoder_model = keras.Model(
    [decoder_sequence_input, encoded_context_input],
    residuals_inf,
    name='decoder_inference_model'
)

In [ ]:
@tf.function(reduce_retracing=True)
def autoregressive_forecast_batch(aux_past, ot_past):
    aux_past = tf.cast(aux_past, tf.float32)
    ot_past = tf.cast(ot_past, tf.float32)

    # Encoder context is computed once.
    encoded_context = encoder_model(
        aux_past,
        training=False
    )

    batch_size = tf.shape(ot_past)[0]

    buffer = tf.concat(
        [
            ot_past,
            tf.zeros(
                (batch_size, PRED_LEN - 1, 1),
                dtype=tf.float32
            )
        ],
        axis=1
    )

    # Fixed residual baseline: last observed OT
    baseline = ot_past[:, -1:, 0:1]

    predictions = []

    for step in range(PRED_LEN):
        residual_outputs = decoder_model(
            [buffer, encoded_context],
            training=False
        )

        output_position = CONTEXT_LEN - 1 + step

        residual = residual_outputs[
            :,
            output_position:output_position + 1,
            :
        ]

        next_value = baseline + residual
        predictions.append(next_value)

        if step < PRED_LEN - 1:
            input_position = CONTEXT_LEN + step

            buffer = tf.concat(
                [
                    buffer[:, :input_position, :],
                    next_value,
                    buffer[:, input_position + 1:, :]
                ],
                axis=1
            )

    return tf.concat(predictions, axis=1)

## 10. Test Set 평가

In [ ]:
EVAL_BATCH_SIZE = 512

pred_batches = []
start_time = time.perf_counter()

for start in range(0, len(X_enc_test), EVAL_BATCH_SIZE):
    end = min(start + EVAL_BATCH_SIZE, len(X_enc_test))

    aux_batch = tf.convert_to_tensor(
        X_enc_test[start:end],
        dtype=tf.float32
    )

    ot_past_batch = tf.convert_to_tensor(
        X_dec_test[start:end, :CONTEXT_LEN, :],
        dtype=tf.float32
    )

    pred_batch = autoregressive_forecast_batch(
        aux_batch,
        ot_past_batch
    ).numpy()

    pred_batches.append(pred_batch[..., 0])

y_pred = np.concatenate(pred_batches, axis=0)

elapsed = time.perf_counter() - start_time
print(f'Autoregressive evaluation time: {elapsed:.2f} s')

y_test_2d = y_test[..., 0]

norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    y_pred.reshape(-1)
)
norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    y_pred.reshape(-1)
)

y_test_real = ot_scaler.inverse_transform(
    y_test_2d.reshape(-1, 1)
).reshape(y_test_2d.shape)

y_pred_real = ot_scaler.inverse_transform(
    y_pred.reshape(-1, 1)
).reshape(y_pred.shape)

real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)
real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    y_pred_real.reshape(-1)
)

print(f'Normalized MSE : {norm_mse:.4f}')
print(f'Normalized MAE : {norm_mae:.4f}')
print(f'MSE (°C²)      : {real_mse:.4f}')
print(f'MAE (°C)       : {real_mae:.4f}')

## 11. Last-Value Baseline

In [ ]:
last_value_pred = np.repeat(
    X_dec_test[:, CONTEXT_LEN - 1, 0][:, None],
    PRED_LEN,
    axis=1
)

baseline_norm_mse = mean_squared_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)
baseline_norm_mae = mean_absolute_error(
    y_test_2d.reshape(-1),
    last_value_pred.reshape(-1)
)

last_value_real = ot_scaler.inverse_transform(
    last_value_pred.reshape(-1, 1)
).reshape(last_value_pred.shape)

baseline_real_mse = mean_squared_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)
baseline_real_mae = mean_absolute_error(
    y_test_real.reshape(-1),
    last_value_real.reshape(-1)
)

print('Last-Value Baseline')
print(f'Normalized MSE : {baseline_norm_mse:.4f}')
print(f'Normalized MAE : {baseline_norm_mae:.4f}')
print(f'MSE (°C²)      : {baseline_real_mse:.4f}')
print(f'MAE (°C)       : {baseline_real_mae:.4f}')

## 12. Decoder-only Residual과 비교

Auxiliary context를 추가한 Encoder–Decoder가
Decoder-only residual model보다 얼마나 개선되는지 확인합니다.

In [ ]:
decoder_only_norm_mse = 0.1070
decoder_only_norm_mae = 0.2551
decoder_only_real_mse = 7.4608
decoder_only_real_mae = 2.1297

print('Decoder-only Residual')
print(f'Normalized MSE : {decoder_only_norm_mse:.4f}')
print(f'Normalized MAE : {decoder_only_norm_mae:.4f}')
print(f'MSE (°C²)      : {decoder_only_real_mse:.4f}')
print(f'MAE (°C)       : {decoder_only_real_mae:.4f}')

print()
print('Encoder-Decoder Residual')
print(f'Normalized MSE : {norm_mse:.4f}')
print(f'Normalized MAE : {norm_mae:.4f}')
print(f'MSE (°C²)      : {real_mse:.4f}')
print(f'MAE (°C)       : {real_mae:.4f}')

## 13. Forecast Example

In [ ]:
sample_idx = 0

past_real = ot_scaler.inverse_transform(
    X_dec_test[sample_idx, :CONTEXT_LEN]
).reshape(-1)

future_real = y_test_real[sample_idx]
pred_real = y_pred_real[sample_idx]

past_x = np.arange(-CONTEXT_LEN + 1, 1)
future_x = np.arange(1, PRED_LEN + 1)

plt.figure(figsize=(10, 4))
plt.plot(past_x, past_real, label='Past OT')
plt.plot(future_x, future_real, label='Ground Truth')
plt.plot(future_x, pred_real, label='Encoder-Decoder Forecast')

plt.axvline(0, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel('OT (°C)')
plt.title('Autoregressive Residual Forecast')
plt.legend()
plt.grid(True)
plt.show()

## 핵심 정리

- Encoder는 auxiliary context를 representation으로 변환합니다.
- Decoder는 causal self-attention으로 OT의 과거를 보고,
  cross-attention으로 encoder context를 참조합니다.
- Cross-attention이 Encoder와 Decoder를 연결합니다.
- 더 복잡한 모델이 항상 simple baseline보다 좋은 것은 아니므로 last-value baseline을 함께 확인합니다.